# Setup

In [ ]:
%pip install picogym[mujoco]
%pip install stable-baselines3[extra] gymnasium

# Training

In [ ]:
from picogym_mujoco.unitree_a1 import UnitreeA1WalkEnv
import numpy as np

from stable_baselines3 import PPO
from gymnasium import spaces, Env


# stable-baselines3 requires wrapping environemnts with gym.Env for training
class WalkerGymWrapper(UnitreeA1WalkEnv, Env):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        obs_high = np.array([np.inf] * self.obs_dim * 2, dtype=np.float32)
        self.observation_space = spaces.Box(-obs_high, obs_high, dtype=np.float32)
        self.action_space = spaces.Box(
            low=self.joint_limits_low,
            high=self.joint_limits_high,
            shape=(12,),
            dtype=np.float32,
        )


reward_weights = {
    "linear_vel_tracking": 10.0,
    "angular_vel_tracking": 0.1,
    "healthy": 1.0,
    "feet_airtime": 20.0,
}
cost_weights = {
    "torque": 0.2,
    "vertical_vel": 0.0,
    "xy_angular_vel": 0.5,
    "action_rate": 0.2,
    "action_sym": 2.5,
    "joint_velocity": 0.01 * 0,
    "joint_acceleration": 2.5e-7 * 0,
    "orientation": 1.0 * 0,
    "collision": 1.0 * 0,
    "default_joint_pos": 0.0,
}

In [ ]:
env = WalkerGymWrapper(reward_weights, cost_weights, headless=True)
policy_kwargs = {"net_arch": dict(pi=[256, 128], vf=[256, 128])}

model = PPO(
    "MlpPolicy",
    env,
    learning_rate=0.001,
    n_steps=512,
    batch_size=64,
    n_epochs=3,
    gamma=0.99,
    gae_lambda=0.92,
    clip_range=0.2,
    ent_coef=0.01,
    policy_kwargs=policy_kwargs,
)

model.learn(
    total_timesteps=100000,
    # progress_bar=True
)

# Simple inference

In [ ]:
import time

env = WalkerGymWrapper(reward_weights, cost_weights, headless=False, notebook=True)
obs, _ = env.reset()

for _ in range(150):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    time.sleep(0.05)

env.close()